In [ ]:
import pandas as pd 
from pace_pic_validation.io.preprocessing import extract_start_date_from_nc
import glob
from pathlib import Path
import xarray as xr
from pace_pic_validation.viz.RGB_functions import rgb_plot_with_stations
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from pace_pic_validation.viz.visualization import plot_rrs_spectra
import re
import os
import pickle
from scipy.spatial import cKDTree
from sklearn.neighbors import BallTree, KDTree
from pace_pic_validation.io.DALEC import openDALEC
from pace_pic_validation.matchup.processing import CL2algorithm

In [ ]:

# datasets 202506
SS202506_merged_ds_dir = '/Volumes/Extreme SSD/Bloom 202506/8D_20250502_20250701'
SS202506_dalec_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/DALEC/ProcDalec_L2_202506_M99.pkl'

# datasets 202606
SS202606_merged_ds_dir = '/Volumes/Extreme SSD/Bloom 202606/merged_ds'


SS202606_dalec_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/DALEC/ProcDalec_L2_202605_M99.pkl'

# all pic insitu data (and lat and lon)
path_all_pic  = '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/all_PIC.csv'


# CHOOSE HERE

merged = SS202506_merged_ds_dir # SS202506_dalec_path
cruise_id = 'SS20250602' #'SS20260601' 'SS20250602'
save_dir = '/Volumes/Extreme SSD/Bloom 202506/timeseries'#'/Volumes/Extreme SSD/Bloom 202606/timeseries'

In [ ]:
# ======= prepare SS stations ======


df = pd.read_csv(path_all_pic)
df = df.sort_values('datetime')

groups = {k: v.copy() for k, v in df.groupby("Campaign")}

SS_campaigns= {k: v for k, v in groups.items() if k.startswith("SS")}

print('available Campaigns')
print(groups.keys())

print('available Simon Stevins Campaigns')
print(SS_campaigns.keys())





# =========== merged Datasets filepath list ========

# SS20250602
paths_merged_ds = glob.glob(merged + '/*.nc')
paths_sorted = sorted(paths_merged_ds)
print(paths_sorted)


In [ ]:
# ====== helper functions ========

# DALEC
from pace_pic_validation.io.DALEC import openDALEC, chl_oc6_dalec, xy_dalec


# PACE

from pace_pic_validation.algorithms.biogeochem import CL2algorithm, chl_pic_oc6_pace_merged
from pace_pic_validation.viz.visualization import pace_granule_plot_prep


# ==============  plotting  ============

plt.style.use("default")
plt.rcParams.update({
    "font.size": 11,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "axes.linewidth": 1.2,
    "axes.labelsize": 12,
    "axes.titlelocation": "left",
    "axes.titleweight": "normal",
    "xtick.major.size": 6,
    "xtick.major.width": 1.2,
    "xtick.minor.size": 3,
    "xtick.minor.width": 1.0,
    "ytick.major.size": 6,
    "ytick.major.width": 1.2,
    "ytick.minor.size": 3,
    "ytick.minor.width": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
    "legend.fontsize": 10,
})

# ---------------------------------------------------------------------------
# 2) Figure-level values
# ---------------------------------------------------------------------------
figsize = (6.5, 4.5)
dpi = 150

# ---------------------------------------------------------------------------
# 3) Axis limits / ticks / labels
# ---------------------------------------------------------------------------
xlim = (400, 700)
ylim = None                 # e.g. (0, 0.02)
xticks = None                # e.g. [400, 450, 500, 550, 600, 650, 700]
xlabel = "Wavelength (nm)"
ylabel = r"Remote-sensing reflectance, $R_{rs}$ (sr$^{-1}$)"

# ---------------------------------------------------------------------------
# 4) Line style kwargs (spectra + DALEC overlay)
# ---------------------------------------------------------------------------
line_kwargs = {}  # e.g. {"linewidth": 0.6, "alpha": 0.85}

dalec_line_kwargs = {
    "color": "black",
    "linestyle": "--",
    "label": "DALEC (Ship-Based)",
}

# ---------------------------------------------------------------------------
# 5) Grid
# ---------------------------------------------------------------------------
grid = True
grid_kwargs = {"ls": ":", "alpha": 0.9}

# ---------------------------------------------------------------------------
# 6) Minor ticks
# ---------------------------------------------------------------------------
minor_ticks = True

# ---------------------------------------------------------------------------
# 7) Spines
# ---------------------------------------------------------------------------
spines_top_right = True

# ---------------------------------------------------------------------------
# 8) Legend
# ---------------------------------------------------------------------------
legend_loc = "upper right"
legend_title = "Station"          # only used in "multi" mode

# ---------------------------------------------------------------------------
# 9) Colormap (multi-station mode)
# ---------------------------------------------------------------------------
cmap = "viridis"
cmap_range = (0, 0.9)
# usage: colors = plt.cm.get_cmap(cmap)(np.linspace(*cmap_range, n_stations))

# ---------------------------------------------------------------------------
# 10) Annotation textbox (Chl-a / PIC values)
# ---------------------------------------------------------------------------
textbox_kwargs = dict(
    x=0.02,
    y=0.08,
    transform_axes=True,       # use ax.transAxes
    fontsize=10,
    verticalalignment="bottom",
    horizontalalignment="left",
    bbox=dict(boxstyle="round", facecolor="none", edgecolor="grey", alpha=0.8),
)

# ---------------------------------------------------------------------------
# 11) Save
# ---------------------------------------------------------------------------
save_kwargs = {}   # e.g. {"dpi": 300}
save_bbox_inches = "tight"


# 

def apply_style_to_ax(ax, title=None):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(loc=legend_loc)
    
    ax.set_ylim(0, 0.015)

    if xlim is not None:
        ax.set_xlim(xlim)
    if xticks is not None:
        ax.set_xticks(xticks)
    if ylim is not None:
        ax.set_ylim(ylim)

    if title is not None:
        ax.set_title(title)

    if grid:
        ax.grid(**grid_kwargs)

    if minor_ticks:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

    if spines_top_right:
        ax.spines["top"].set_visible(True)
        ax.spines["right"].set_visible(True)

In [ ]:
def plot_rrs_pace_L3(path, lat, lon, label, save_dir):

    ds = xr.open_dataset(path)

    ds_pt = ds['Rrs'].sel(
                        longitude=lon,
                        latitude=lat,
                        method="nearest",
                    )

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    ax.plot(
            ds_pt['wavelength'].values,
            ds_pt.values,
            **line_kwargs,
            label='PACE',
                )    

    apply_style_to_ax(ax, title=label)
    
    plt.tight_layout()

    save_path = os.path.join(save_dir, f"{label}_rrs_spectrum.png")
    fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
    
    return fig 



import os
import re
import xarray as xr
import matplotlib.pyplot as plt


import os
import re
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np


def plot_rrs_pace_L3_station(paths, lat, lon, station, campaign_name, save_dir, plot_name,
                              cmap="viridis", cmap_range=(0, 0.9)):
    """
    Plot Rrs spectra for a single station across multiple PACE L3 granules
    (one line per date/path), all on one figure, saved under the campaign name.
    Lines are colored on a gradient (early -> late) so temporal change is visible.

    Parameters
    ----------
    paths : sequence of str
        Sorted granule paths to overlay (one line each), assumed chronological.
    lat, lon : float
        Station coordinates.
    station : str
        Station name (used only for the plot title, not the filename).
    campaign_name : str
        Used as the saved filename (e.g. "SS20250602").
    save_dir : str
        Directory to save the figure into (typically the station's folder).
    cmap : str, optional
        Matplotlib colormap name for the time gradient.
    cmap_range : tuple, optional
        Fractional range of colormap to use, e.g. (0, 0.9).

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    colors = plt.cm.get_cmap(cmap)(np.linspace(*cmap_range, len(paths)))

    for i, path in enumerate(paths):
        match = re.search(r"\d{4}-\d{2}-\d{2}_\d{4}-\d{2}-\d{2}", path)
        date_label = match.group(0) if match else os.path.basename(path)
        week_label = f"Week {i + 1}"

        ds = xr.open_dataset(path)
        ds_pt = ds['Rrs'].sel(
            longitude=lon,
            latitude=lat,
            method="nearest",
        )

        ax.plot(
            ds_pt['wavelength'].values,
            ds_pt.values,
            color=colors[i],
            label=week_label,
        )

    apply_style_to_ax(ax, title=station)

    plt.tight_layout()

    save_path = os.path.join(save_dir, f"{campaign_name}_{plot_name}_rrs_spectrum.png")
    fig.savefig(save_path, dpi=dpi, bbox_inches="tight")

    return fig

In [ ]:
df = SS_campaigns[cruise_id]

df.to_csv('/Users/timfugl/Desktop/mail ivona/data/cruise_data/SS20260602.csv')

print(df.columns)

lats = list(df['lat'])
lons = list(df['lon'])
stations = list(df['Station'])

bloom_dir = save_dir

for path in paths_sorted:
    print(path)
    match = re.search(r"\d{4}-\d{2}-\d{2}_\d{4}-\d{2}-\d{2}", path)
    
    if match:
        date = match.group(0)
  

    for lat, lon, station in zip(lats, lons, stations):
        
        station_dir = os.path.join(bloom_dir, station)
        os.makedirs(station_dir, exist_ok=True)

        save_dir = station_dir

        label = f"{station}_{date}"

        plot_rrs_pace_L3(path, lat, lon, label, save_dir)


In [ ]:
campaign_name = cruise_id
df = SS_campaigns[campaign_name]
print(df.columns)

lats = list(df['lat'])
lons = list(df['lon'])
stations = list(df['Station'])

bloom_dir = save_dir
for lat, lon, station in zip(lats, lons, stations):

    station_dir = os.path.join(bloom_dir, station)
    os.makedirs(station_dir, exist_ok=True)

    plot_rrs_pace_L3_station(
        paths_sorted, lat, lon, station, campaign_name, save_dir=station_dir, cmap= 'viridis_r',plot_name = station
    )